# Code to create train and validation data splits

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from Bio import SeqIO

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

from joblib import load

from Bio.Cluster import kcluster
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import pearsonr, spearmanr

import os
os.chdir("/Users/claireleblanc/Documents/grad_school/staller_lab/NN_interpretability_for_AD_prediction/Model")
from ADModel_two_state import ADModel_two_state_abund
from Data import DataReader, SplitData, FastTensorDataLoader, one_hot_encode

os.chdir("/Users/claireleblanc/Documents/grad_school/staller_lab/NN_interpretability_for_AD_prediction/Data")

In [2]:
# Parameters to make figures work better with illustrator 
import matplotlib 
# For higher resoltion figures
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Helvetica" #somethings this one doesnt work
plt.rcParams['pdf.fonttype'] = 42

In [3]:
# Read in the tile data --> Contains the name of the sequence (will be used later)
tile_data = pd.read_csv("Gcn4Array_Design.csv", index_col=0)
tile_data

,ADseq,Name,ArrayDNA
0,MALRIEVYNRIESSTASTALQRQDLRYTFRSNARAASGQA,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_1,ATGGCTTTGAGAATTGAAGTTTATAATAGAATTGAATCTTCTACTG...
1,EVYNRIESSTASTALQRQDLRYTFRSNARAASGQANANYQ,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_6,GAAGTTTATAATAGAATTGAATCTTCTACTGCTTCTACTGCTTTGC...
2,IESSTASTALQRQDLRYTFRSNARAASGQANANYQAFTAG,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_11,ATTGAATCTTCTACTGCTTCTACTGCTTTGCAAAGACAAGATTTGA...
3,ASTALQRQDLRYTFRSNARAASGQANANYQAFTAGSALNG,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_16,GCTTCTACTGCTTTGCAAAGACAAGATTTGAGATATACATTTAGAT...
4,QRQDLRYTFRSNARAASGQANANYQAFTAGSALNGPSLPA,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_21,CAAAGACAAGATTTGAGATATACATTTAGATCTAATGCTAGAGCTG...
...,...,...,...
47,AKVDTEEEDKTMVDSTSLSWEDLFDFESYSTDLIASINPD,Gnc4Lib_Kappa_HIGH_0,GCTAAAGTTGATACTGAAGAAGAAGATAAGACTATGGTTGATTCTA...
48,STDYTPMFEYETYEDNFKEWTSLFDNDIPVTTDDVSLADR,Gnc4Lib_Disorder_HIGH_0,TCTACTGATTATACTCCAATGTTTGAATATGAAACTTATGAAGATA...
49,MTDSTPMFEYELTENNSKEWTSLFDTDIPVTTDDESLADK,Gnc4Lib_Disorder_HIGH_0,ATGACTGATTCTACTCCAATGTTTGAATATGAATTGACTGAGAATA...
50,STDSTPMFEYQNLENNSKEWTSLFDNDIPVTTDNVSLADK,Gnc4Lib_Charge_HIGH_0,TCTACTGATTCTACTCCAATGTTTGAATATCAGAATTTGGAGAATA...


In [4]:
# Read in the full length sequences as a fasta file
input_file = "Unique_502_Gcn4_Seqs.fasta"
unqiue_seqs = SeqIO.parse(open(input_file), 'fasta')
ids = []
seqs = []
for seq in unqiue_seqs:
    ids.append(seq.id)
    seqs.append(str(seq.seq))


In [5]:
# Turn the full length sequences into a dataframe
unique_gcn4 = pd.DataFrame([ids,seqs]).T
unique_gcn4.columns = ["Name", "Sequence"]

In [6]:
# This is a file calculating the distance between all pairs of full length sequences
file_name = "Unique_502_Gcn4_Seqs.fasta.hat2"

# Read in the data
with open(file_name, "r") as f:
    data = f.readlines()

# Second line is number of seqs
num_seqs = int(data[1].strip())

# Initalize an empty distance matrix
dist_mtx = np.zeros((num_seqs, num_seqs))


i = 3 + num_seqs # Skip the header rows --> There are three header lines and then a line with the name of each sequence
j = 1 # Counting the columns

while j < (num_seqs):
    row = []

    # Distances for a single row in the distance matrix span multiple rows in the hat2 file
    # This reads through a single row in the distance matrix
    # hat2 file only contains half distance matrix (because it is symetrical)
    while len(row) < (num_seqs-j):
        # Each ith row gives list of numbers, transform them to flaots
        row += [float(d) for d in data[i].strip().split()] 
        i += 1

    # Add row to distance matrix
    dist_mtx[j-1, j:j+len(row)] = row
    j += 1

# Fills in opposite half of distance matrix
dist_mtx = dist_mtx + dist_mtx.T

In [7]:
# Cluster the sequences based on the distance matrix
kcluster_result = kcluster(dist_mtx)[0]
heirarchical_clustering = linkage(dist_mtx, method='single')
heirarchical_result = fcluster(heirarchical_clustering, 3, criterion="maxclust") # End is the level, closer to end means less clusters

/var/folders/34/1pw3__x51kdfyh3vk72f5vc80000gn/T/ipykernel_26658/1712118893.py:3: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  heirarchical_clustering = linkage(dist_mtx, method='single')


In [8]:
K = 3
Z = heirarchical_clustering
# height halfway between the merges that go from K to K-1 clusters
cut = (Z[-K, 2] + Z[-K+1, 2]) / 2.0

In [9]:
unique_gcn4["kcluster"] = kcluster_result
unique_gcn4["heirarchical"] = heirarchical_result
unique_gcn4["heirarchical"].value_counts()

heirarchical
3    415
1     63
2     24
Name: count, dtype: int64

In [ ]:
# Saving cluster results to csv
# unique_gcn4.to_csv("Unique_502_Gcn4_Seqs_three_clusters.csv")

In [10]:
training_set = unique_gcn4[unique_gcn4["heirarchical"] == 3]
validation_set = unique_gcn4[unique_gcn4["heirarchical"] == 2]
test_set = unique_gcn4[unique_gcn4["heirarchical"] == 1]

In [11]:
# Getting the species each tile comes from
tile_data["Species"] = [re.sub(r'_[0-9]*$' ,"",i) for i in tile_data["Name"]]
tile_data

,ADseq,Name,ArrayDNA,Species
0,MALRIEVYNRIESSTASTALQRQDLRYTFRSNARAASGQA,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_1,ATGGCTTTGAGAATTGAAGTTTATAATAGAATTGAATCTTCTACTG...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
1,EVYNRIESSTASTALQRQDLRYTFRSNARAASGQANANYQ,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_6,GAAGTTTATAATAGAATTGAATCTTCTACTGCTTCTACTGCTTTGC...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
2,IESSTASTALQRQDLRYTFRSNARAASGQANANYQAFTAG,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_11,ATTGAATCTTCTACTGCTTCTACTGCTTTGCAAAGACAAGATTTGA...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
3,ASTALQRQDLRYTFRSNARAASGQANANYQAFTAGSALNG,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_16,GCTTCTACTGCTTTGCAAAGACAAGATTTGAGATATACATTTAGAT...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
4,QRQDLRYTFRSNARAASGQANANYQAFTAGSALNGPSLPA,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_21,CAAAGACAAGATTTGAGATATACATTTAGATCTAATGCTAGAGCTG...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
...,...,...,...,...
47,AKVDTEEEDKTMVDSTSLSWEDLFDFESYSTDLIASINPD,Gnc4Lib_Kappa_HIGH_0,GCTAAAGTTGATACTGAAGAAGAAGATAAGACTATGGTTGATTCTA...,Gnc4Lib_Kappa_HIGH
48,STDYTPMFEYETYEDNFKEWTSLFDNDIPVTTDDVSLADR,Gnc4Lib_Disorder_HIGH_0,TCTACTGATTATACTCCAATGTTTGAATATGAAACTTATGAAGATA...,Gnc4Lib_Disorder_HIGH
49,MTDSTPMFEYELTENNSKEWTSLFDTDIPVTTDDESLADK,Gnc4Lib_Disorder_HIGH_0,ATGACTGATTCTACTCCAATGTTTGAATATGAATTGACTGAGAATA...,Gnc4Lib_Disorder_HIGH
50,STDSTPMFEYQNLENNSKEWTSLFDNDIPVTTDNVSLADK,Gnc4Lib_Charge_HIGH_0,TCTACTGATTCTACTCCAATGTTTGAATATCAGAATTTGGAGAATA...,Gnc4Lib_Charge_HIGH


In [12]:
train_tiles = tile_data[[i in set(training_set["Name"]) for i in tile_data["Species"]]]
train_tiles

,ADseq,Name,ArrayDNA,Species
0,MALRIEVYNRIESSTASTALQRQDLRYTFRSNARAASGQA,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_1,ATGGCTTTGAGAATTGAAGTTTATAATAGAATTGAATCTTCTACTG...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
1,EVYNRIESSTASTALQRQDLRYTFRSNARAASGQANANYQ,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_6,GAAGTTTATAATAGAATTGAATCTTCTACTGCTTCTACTGCTTTGC...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
2,IESSTASTALQRQDLRYTFRSNARAASGQANANYQAFTAG,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_11,ATTGAATCTTCTACTGCTTCTACTGCTTTGCAAAGACAAGATTTGA...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
3,ASTALQRQDLRYTFRSNARAASGQANANYQAFTAGSALNG,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_16,GCTTCTACTGCTTTGCAAAGACAAGATTTGAGATATACATTTAGAT...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
4,QRQDLRYTFRSNARAASGQANANYQAFTAGSALNGPSLPA,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g_21,CAAAGACAAGATTTGAGATATACATTTAGATCTAATGCTAGAGCTG...,Sordariomycetes_jgi|Acral2|2019554|gm1.4974_g
...,...,...,...,...
71727,VAVKRARNTMAARKSRRRKLEKQEQMEDRIRELEAMLAKS,Tmar_XP_002151511.1_TalaromycesMarneffeiATCC18...,GTTGCTGTTAAGAGAGCTAGAAATACTATGGCTGCTAGGAAATCTA...,Tmar_XP_002151511.1_TalaromycesMarneffeiATCC18224
71728,ARNTMAARKSRRRKLEKQEQMEDRIRELEAMLAKSEKDVQ,Tmar_XP_002151511.1_TalaromycesMarneffeiATCC18...,GCTAGAAATACTATGGCTGCTAGGAAATCTAGAAGAAGGAAATTGG...,Tmar_XP_002151511.1_TalaromycesMarneffeiATCC18224
71729,AARKSRRRKLEKQEQMEDRIRELEAMLAKSEKDVQYWKAM,Tmar_XP_002151511.1_TalaromycesMarneffeiATCC18...,GCTGCTAGGAAATCTAGAAGAAGGAAATTGGAGAAACAAGAACAAA...,Tmar_XP_002151511.1_TalaromycesMarneffeiATCC18224
71730,RRRKLEKQEQMEDRIRELEAMLAKSEKDVQYWKAMAQTSM,Tmar_XP_002151511.1_TalaromycesMarneffeiATCC18...,AGAAGAAGGAAATTGGAGAAACAAGAACAAATGGAAGATAGAATTA...,Tmar_XP_002151511.1_TalaromycesMarneffeiATCC18224


In [13]:
val_tiles = tile_data[[i in set(validation_set["Name"]) for i in tile_data["Species"]]]
val_tiles

,ADseq,Name,ArrayDNA,Species
34057,MSGRVSSPPVTVPPRDLFGGLESSTPWIREQERIRKKPLL,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653...,ATGTCTGGTAGAGTTTCTTCTCCACCAGTTACTGTTCCACCAAGAG...,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653_g
34058,SSPPVTVPPRDLFGGLESSTPWIREQERIRKKPLLVADLQ,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653...,TCTTCTCCACCAGTTACTGTTCCACCAAGAGATTTGTTTGGTGGTT...,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653_g
34059,TVPPRDLFGGLESSTPWIREQERIRKKPLLVADLQYVEGL,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653...,ACTGTTCCACCAAGAGATTTGTTTGGTGGTTTGGAATCTTCTACTC...,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653_g
34060,DLFGGLESSTPWIREQERIRKKPLLVADLQYVEGLRWVTW,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653...,GATTTGTTTGGTGGTTTGGAATCTTCTACTCCATGGATTAGAGAAC...,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653_g
34061,LESSTPWIREQERIRKKPLLVADLQYVEGLRWVTWGQMGH,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653...,TTGGAATCTTCTACTCCATGGATTAGAGAACAAGAAAGAATTAGGA...,Lecanoromycetes_jgi|LobpulSc1|2154466|gm1.7653_g
...,...,...,...,...
62790,EGTSTPGGSSSPEPELTVTLPDAHPDTMFDFNHNDFISVP,Lecanoromycetes_jgi|Usnflo1|874927|fgenesh1_kg...,GAAGGTACTTCTACTCCAGGTGGTTCTTCTTCTCCAGAACCAGAAT...,Lecanoromycetes_jgi|Usnflo1|874927|fgenesh1_kg...
62791,PGGSSSPEPELTVTLPDAHPDTMFDFNHNDFISVPESSDA,Lecanoromycetes_jgi|Usnflo1|874927|fgenesh1_kg...,CCAGGTGGTTCTTCTTCTCCAGAACCAGAATTGACTGTTACTTTGC...,Lecanoromycetes_jgi|Usnflo1|874927|fgenesh1_kg...
62792,SPEPELTVTLPDAHPDTMFDFNHNDFISVPESSDALFDES,Lecanoromycetes_jgi|Usnflo1|874927|fgenesh1_kg...,TCTCCAGAACCAGAATTGACTGTTACTTTGCCAGATGCTCATCCAG...,Lecanoromycetes_jgi|Usnflo1|874927|fgenesh1_kg...
62793,LTVTLPDAHPDTMFDFNHNDFISVPESSDALFDESLDYGT,Lecanoromycetes_jgi|Usnflo1|874927|fgenesh1_kg...,TTGACTGTTACTTTGCCAGATGCTCATCCAGATACTATGTTTGACT...,Lecanoromycetes_jgi|Usnflo1|874927|fgenesh1_kg...


In [14]:
test_tiles = tile_data[[i in set(test_set["Name"]) for i in tile_data["Species"]]]
test_tiles

,ADseq,Name,ArrayDNA,Species
5379,MSTPNIAQDFPELFDLQSNRFGDDFSSPESTMLSPQISNS,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...,ATGTCTACTCCAAATATTGCTCAAGACTTTCCAGAATTGTTTGATT...,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...
5380,IAQDFPELFDLQSNRFGDDFSSPESTMLSPQISNSIFSQM,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...,ATTGCTCAAGACTTTCCAGAATTGTTTGATTTGCAATCTAATAGAT...,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...
5381,PELFDLQSNRFGDDFSSPESTMLSPQISNSIFSQMGDAAP,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...,CCAGAATTGTTTGATTTGCAATCTAATAGATTTGGTGATGACTTCT...,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...
5382,LQSNRFGDDFSSPESTMLSPQISNSIFSQMGDAAPPGTVS,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...,TTGCAATCTAATAGATTTGGTGATGACTTCTCTTCTCCAGAATCTA...,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...
5383,FGDDFSSPESTMLSPQISNSIFSQMGDAAPPGTVSPRDLF,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...,TTTGGTGATGACTTCTCTTCTCCAGAATCTACTATGTTGTCTCCAC...,Eurotiomycetes_jgi|Aspca3|130068|estExt_Genema...
...,...,...,...,...
71682,VDPSDPVALKRARNTEAARKSRARKLERQDEMERRIRELE,Nfis_EAW24893.1_NeosartoryaFischeriNRRL181_191,GTTGATCCATCTGATCCAGTTGCTTTGAAGAGAGCTAGAAATACTG...,Nfis_EAW24893.1_NeosartoryaFischeriNRRL181
71683,PVALKRARNTEAARKSRARKLERQDEMERRIRELEKSLEE,Nfis_EAW24893.1_NeosartoryaFischeriNRRL181_196,CCAGTTGCTTTGAAGAGAGCTAGAAATACTGAAGCTGCTAGGAAAT...,Nfis_EAW24893.1_NeosartoryaFischeriNRRL181
71684,RARNTEAARKSRARKLERQDEMERRIRELEKSLEEAQQRE,Nfis_EAW24893.1_NeosartoryaFischeriNRRL181_201,AGAGCTAGAAATACTGAAGCTGCTAGGAAATCTAGAGCTAGGAAAT...,Nfis_EAW24893.1_NeosartoryaFischeriNRRL181
71685,EAARKSRARKLERQDEMERRIRELEKSLEEAQQREQYWKA,Nfis_EAW24893.1_NeosartoryaFischeriNRRL181_206,GAAGCTGCTAGGAAATCTAGAGCTAGGAAATTGGAAAGACAAGATG...,Nfis_EAW24893.1_NeosartoryaFischeriNRRL181


In [15]:
train_tiles[[s == "LTALTSPSLFDGSPDFDTFDISPNFGHSDLENPDTWFSLF" for s in train_tiles["ADseq"]]]

,ADseq,Name,ArrayDNA,Species
41939,LTALTSPSLFDGSPDFDTFDISPNFGHSDLENPDTWFSLF,Sordariomycetes_jgi|Neucr4830_1|444994|fgenesh...,TTGACTGCTTTGACTTCTCCATCTTTGTTTGATGGTTCTCCAGACT...,Sordariomycetes_jgi|Neucr4830_1|444994|fgenesh...


In [16]:
other_tiles = tile_data[[(i not in set(validation_set["Name"])) and (i not in set(training_set["Name"])) and (i not in set(test_set["Name"])) for i in tile_data["Species"]]]
other_tiles

,ADseq,Name,ArrayDNA,Species
0,STIPLDFMPRDALHGFDWSEEDDMSDGLPFLKTDPNNNGF,GAL4_AD1_0,TCTACTATTCCATTGGACTTTATGCCAAGAGATGCTTTGCATGGAT...,GAL4_AD1
1,STIPLDAMPRDALHGADASEEDDMSDGLPALKTDPNNNGA,GAL4_AD1_Aro2A_0,TCTACTATTCCATTGGATGCTATGCCAAGAGATGCTTTGCATGGTG...,GAL4_AD1_Aro2A
2,STIPADFMPRDAAHGFDWSEEDDMSDGAPFAKTDPNNNGF,GAL4_AD1_L2A_0,TCTACTATTCCAGCTGACTTTATGCCAAGAGATGCTGCTCATGGAT...,GAL4_AD1_L2A
3,STDSTPMFEYENLEDNSKEWTSLFDNDIPVTTDDVSLADK,GCN4_CAAD40_0,TCTACTGATTCTACTCCAATGTTTGAATATGAGAATTTGGAAGATA...,GCN4_CAAD40
4,STDSTPMFEYENLEDNSKEATSAADNDIPVTTDDVSLADK,GCN4_CAAD40_WLF_A_0,TCTACTGATTCTACTCCAATGTTTGAATATGAGAATTTGGAAGATA...,GCN4_CAAD40_WLF_A
...,...,...,...,...
47,AKVDTEEEDKTMVDSTSLSWEDLFDFESYSTDLIASINPD,Gnc4Lib_Kappa_HIGH_0,GCTAAAGTTGATACTGAAGAAGAAGATAAGACTATGGTTGATTCTA...,Gnc4Lib_Kappa_HIGH
48,STDYTPMFEYETYEDNFKEWTSLFDNDIPVTTDDVSLADR,Gnc4Lib_Disorder_HIGH_0,TCTACTGATTATACTCCAATGTTTGAATATGAAACTTATGAAGATA...,Gnc4Lib_Disorder_HIGH
49,MTDSTPMFEYELTENNSKEWTSLFDTDIPVTTDDESLADK,Gnc4Lib_Disorder_HIGH_0,ATGACTGATTCTACTCCAATGTTTGAATATGAATTGACTGAGAATA...,Gnc4Lib_Disorder_HIGH
50,STDSTPMFEYQNLENNSKEWTSLFDNDIPVTTDNVSLADK,Gnc4Lib_Charge_HIGH_0,TCTACTGATTCTACTCCAATGTTTGAATATCAGAATTTGGAGAATA...,Gnc4Lib_Charge_HIGH


In [17]:
train_tiles[["Blastocladiomycota_jgi|Catan2|1506241|gm1.1155" in s for s in train_tiles["Name"]]]

,ADseq,Name,ArrayDNA,Species
13799,MPLTFAIITMASTPDFARLDDPSVRLPDSVSVPFGSGSTP,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,ATGCCATTGACATTTGCTATTATTACTATGGCTTCTACTCCAGACT...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g
13800,AIITMASTPDFARLDDPSVRLPDSVSVPFGSGSTPSALFP,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,GCTATTATTACTATGGCTTCTACTCCAGACTTTGCTAGATTGGATG...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g
13801,ASTPDFARLDDPSVRLPDSVSVPFGSGSTPSALFPKSLLS,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,GCTTCTACTCCAGACTTTGCTAGATTGGATGATCCATCTGTTAGAT...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g
13802,FARLDDPSVRLPDSVSVPFGSGSTPSALFPKSLLSLPLVP,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,TTTGCTAGATTGGATGATCCATCTGTTAGATTGCCAGATTCTGTTT...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g
13803,DPSVRLPDSVSVPFGSGSTPSALFPKSLLSLPLVPRPAQA,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,GACCCATCTGTTAGATTGCCAGATTCTGTTTCTGTTCCATTTGGTT...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g
...,...,...,...,...
13879,RKQAKLEYLERHVGELEDVNARLKRQVDLLRRKVDEGNKL,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,AGGAAACAAGCTAAATTGGAATATTTGGAAAGACATGTTGGTGAAT...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g
13880,LEYLERHVGELEDVNARLKRQVDLLRRKVDEGNKLAAQGH,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,TTGGAATATTTGGAAAGACATGTTGGTGAATTGGAAGATGTTAATG...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g
13881,RHVGELEDVNARLKRQVDLLRRKVDEGNKLAAQGHQHLDV,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,AGACATGTTGGTGAATTGGAAGATGTTAATGCTAGATTGAAGAGAC...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g
13882,LEDVNARLKRQVDLLRRKVDEGNKLAAQGHQHLDVFGGCR,Blastocladiomycota_jgi|Catan2|1506241|gm1.1155...,TTGGAAGATGTTAATGCTAGATTGAAGAGACAAGTTGATTTGTTGA...,Blastocladiomycota_jgi|Catan2|1506241|gm1.11555_g


In [19]:
# Loading in the tiles that we actually have experimental results for
experiment_data = pd.read_csv("pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio.csv")
experiment_data

,aa_seq,activity,abundance,ratio
0,LQDFVLFDQPIRPHRQHNRNALQPPTRGINLNQQHRSQHL,270.500000,2275.157246,23705.971880
1,DLFGGLESSTPWIREQERIHLQYVEGLRWVTWGQMGHVEL,478.640464,4281.992228,7965.856073
2,VEHSPAEKSDDLEVVEPTSGHQRRKSGTSPPSGRHSSVSG,1081.420642,3180.013344,18161.451312
3,NAGTASRFLTTVVALCSPSDVSSTVLTGNARMQVRPIGPL,603.933187,1160.105019,52119.378755
4,QQQHRPHSTLQASSASPIQNPRVSDLSQDTGSIASSTSPQ,425.945104,3136.116978,19044.671497
...,...,...,...,...
17727,LYDESPDFGSGFDVSPNFAGSDFDAGGNDVWFPLFPQSNT,8178.009196,3209.784417,262143.000000
17728,HANRGPDFDALFDLTANSFVDGLDAASLAMFDTQQLDKVQ,9064.000000,4649.000000,262143.000000
17729,LYESPDFGYDVSPGFGSNDFDTGSNQWFSLFPDQSTTPDA,9064.000000,4649.000000,262143.000000
17730,TPNIPQEFFDFTEGFGEEFTDSTMLSPHLVPTGIMASKDS,4862.343223,111.000000,262143.000000


In [20]:
trainning_set = set(train_tiles["ADseq"])
training_data = experiment_data[[i in  trainning_set for i in experiment_data["aa_seq"]]]
training_data

,aa_seq,activity,abundance,ratio
0,LQDFVLFDQPIRPHRQHNRNALQPPTRGINLNQQHRSQHL,270.500000,2275.157246,23705.971880
2,VEHSPAEKSDDLEVVEPTSGHQRRKSGTSPPSGRHSSVSG,1081.420642,3180.013344,18161.451312
3,NAGTASRFLTTVVALCSPSDVSSTVLTGNARMQVRPIGPL,603.933187,1160.105019,52119.378755
5,SSTTALQQQHRQTRPQVPLFSQSTGSIPKTPNMVMQGTYI,706.024331,1005.046690,55857.888067
6,KGGKLQASKEALYLGNAGTASRFLTTVVALCSPSDVSSTV,474.552224,1537.143665,32333.990842
...,...,...,...,...
17725,PTPTFSSPYLFDSPSEGYETSPLFGAEDTNGDNWYSLFPE,8343.122503,1971.676200,262143.000000
17726,DAWFSLFPSISGGENDESPLSAAEDVMVADPFNISAQVVL,4633.269589,280.000000,262143.000000
17727,LYDESPDFGSGFDVSPNFAGSDFDAGGNDVWFPLFPQSNT,8178.009196,3209.784417,262143.000000
17729,LYESPDFGYDVSPGFGSNDFDTGSNQWFSLFPDQSTTPDA,9064.000000,4649.000000,262143.000000


In [21]:
for i in range(400, 1000):
    shuffled_training_data = training_data.sample(frac=1, random_state=i)
    shuffled_training_data['aa_seq'] = training_data['aa_seq'].values
    shuffled_training_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_training_shuffled_{i}.csv",index=False)

In [22]:
val_set = set(val_tiles["ADseq"])
validation_data = experiment_data[[((i in val_set)) for i in experiment_data["aa_seq"]]]
validation_data

,aa_seq,activity,abundance,ratio
1,DLFGGLESSTPWIREQERIHLQYVEGLRWVTWGQMGHVEL,478.640464,4281.992228,7965.856073
4,QQQHRPHSTLQASSASPIQNPRVSDLSQDTGSIASSTSPQ,425.945104,3136.116978,19044.671497
25,NPRVSDLSQDTGSIASSTSPQQSNPTGQQHHFYASSAPSS,2030.987086,2681.865581,18231.310507
123,PSIQQFNSPTGQQQRFYANSAPSSTTGLHQQSPRSRPPVP,285.000000,3506.279714,19522.323058
135,QNPRVSDLSQDTGSIASSTSPQQSNPTGQQHHFYASSAPS,486.971114,3670.298469,15030.686719
...,...,...,...,...
17590,MLGLFISTCILIAPLYLVYKPPVYLVRYFQWRWPDVLWCV,256.000000,280.000000,82477.974548
17596,PGTSYIDSPLYIADSTDTSPLFATDGLGADADSWAPLFND,6874.030718,3018.894207,199646.447487
17608,RALPNSSVLTLPSSDMLFDLGDFPASPDVSPDVSFYSPAD,5858.992902,4183.831469,156966.965242
17621,VQTISPKDIMMDTMSAPPSTTYTNLTTPGTSYIDSPLYIA,3492.793674,700.618016,200577.344363


In [23]:
for i in range(400, 1000):
    shuffled_val_data = validation_data.sample(frac=1, random_state=i)
    shuffled_val_data['aa_seq'] = validation_data['aa_seq'].values
    shuffled_val_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_validation_shuffled_{i}.csv",index=False)

In [24]:
test_set = set(test_tiles["ADseq"])
other_set = set(other_tiles["ADseq"])
test_data = experiment_data[[(i in test_set or (i in other_set)) for i in experiment_data["aa_seq"]]]
test_data

,aa_seq,activity,abundance,ratio
14,GPLFPAQDDFSTAFDSAALDAAIALSQPETIPAKEISVPP,3112.003616,1714.880232,120898.952542
24,TNVIAQAQGYRPSSHRLSLPGPARLSQRQHIYAASDPSNS,437.436271,2155.484437,35747.526907
51,PPTDVSAGDEAHADGEDVAMAHADAADDFDADMAGDGDSP,579.237129,3566.691915,17543.166342
55,SAFDSAALDVALALSQPETKPAKEVSVPPSPAIRNSASPA,408.878538,3013.348811,30430.118200
61,SSTNLDDLSAAVSAPPKTVPPPPSPMVRAASSPGQSTGTS,605.266583,3117.194936,23469.400978
...,...,...,...,...
17714,PGYFSQDTSPMFATDMELGPGVEEWGSLFPSQDDFSLGLD,8812.183991,2964.401324,262143.000000
17721,YFSQDTSPMFPTDLELNPGHEEWDSLFPPQDGFPVAFDSA,8709.017129,111.000000,262143.000000
17723,PDTLRLDLRTNFDDPEFFDFTEGFGEEFTDSTMLSPHLVP,9064.000000,1535.915898,262143.000000
17728,HANRGPDFDALFDLTANSFVDGLDAASLAMFDTQQLDKVQ,9064.000000,4649.000000,262143.000000


In [25]:
for i in range(400, 1000):
    shuffled_test_data = test_data.sample(frac=1, random_state=i)
    shuffled_test_data['aa_seq'] = test_data['aa_seq'].values
    shuffled_test_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_test_shuffled_{i}.csv",index=False)

# Shuffling abundance only

In [36]:
# Shuffle everything, then rewrit aa and activity columns
for i in range(0, 400):
    shuffled_training_data = training_data.sample(frac=1, random_state=i)
    shuffled_training_data['aa_seq'] = training_data['aa_seq'].values
    shuffled_training_data['activity'] = training_data['activity'].values
    shuffled_training_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_training_shuffled_abund_only_{i}.csv",index=False)

In [37]:
# Shuffle everything, then rewrit aa and activity columns
for i in range(0, 400):
    shuffled_val_data = validation_data.sample(frac=1, random_state=i)
    shuffled_val_data['aa_seq'] = validation_data['aa_seq'].values
    shuffled_val_data['activity'] = validation_data['activity'].values
    shuffled_val_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_validation_shuffled_abund_only_{i}.csv",index=False)

In [38]:
# Shuffle everything, then rewrit aa and activity columns
for i in range(0, 400):
    shuffled_test_data = test_data.sample(frac=1, random_state=i)
    shuffled_test_data['aa_seq'] = test_data['aa_seq'].values
    shuffled_test_data['activity'] = test_data['activity'].values
    shuffled_test_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_test_shuffled_abund_only_{i}.csv",index=False)

# Shuffling activity only

In [33]:
# Shuffle everything, then rewrit aa and abund columns
for i in range(0, 400):
    shuffled_training_data = training_data.sample(frac=1, random_state=i)
    shuffled_training_data['aa_seq'] = training_data['aa_seq'].values
    shuffled_training_data['abundance'] = training_data['abundance'].values
    shuffled_training_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_training_shuffled_act_only_{i}.csv",index=False)

In [34]:
# Shuffle everything, then rewrit aa and activity columns
for i in range(0, 400):
    shuffled_val_data = validation_data.sample(frac=1, random_state=i)
    shuffled_val_data['aa_seq'] = validation_data['aa_seq'].values
    shuffled_val_data['abundance'] = validation_data['abundance'].values
    shuffled_val_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_validation_shuffled_act_only_{i}.csv",index=False)

In [35]:
# Shuffle everything, then rewrit aa and activity columns
for i in range(0, 400):
    shuffled_test_data = test_data.sample(frac=1, random_state=i)
    shuffled_test_data['aa_seq'] = test_data['aa_seq'].values
    shuffled_test_data['abundance'] = test_data['abundance'].values
    shuffled_test_data.to_csv(f"shuffles/pm_gcn4_sort2_pools_allchannels_wrangled_w_ratio_test_shuffled_act_only_{i}.csv",index=False)